In [1]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.1 MB/s eta 0:00:00


In [2]:
import sqlite3
import random
from faker import Faker
import numpy as np

In [3]:
conn = sqlite3.connect("retail_store.db")
cursor = conn.cursor()

In [4]:
cursor.execute("""
CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    gender TEXT,
    membership_level TEXT CHECK(membership_level IN ('Bronze','Silver','Gold','Platinum')),
    date_of_birth DATE,
    email TEXT,
    annual_income REAL
);
""")

In [5]:
cursor.execute("""
CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL CHECK(price > 0),
    stock_quantity INTEGER,
    supplier_rating REAL,
    weight_grams REAL
);
""")

In [6]:
cursor.execute("""
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    shipping_method TEXT,
    order_status TEXT CHECK(order_status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
);
""")

In [7]:
cursor.execute("""
CREATE TABLE Order_Items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER CHECK(quantity > 0),
    unit_price REAL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES Orders(order_id),
    FOREIGN KEY (product_id) REFERENCES Products(product_id)
);
""")

In [8]:
cursor.execute("""
CREATE TABLE Payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    payment_method TEXT,
    payment_date DATE,
    amount_paid REAL,
    payment_status TEXT,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id)
);
""")

In [9]:
cursor.execute("""
CREATE TABLE Reviews (
    review_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    customer_id INTEGER,
    rating INTEGER CHECK(rating BETWEEN 1 AND 5),
    review_text TEXT,
    review_date DATE,
    FOREIGN KEY (product_id) REFERENCES Products(product_id),
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
);
""")

In [10]:
conn.commit()

In [11]:
fake = Faker()

membership_levels = ["Bronze","Silver","Gold","Platinum"]

for i in range(1, 501):
    income = round(np.random.normal(35000, 10000),2)

    # 5% missing income
    if random.random() < 0.05:
        income = None

    cursor.execute("""
    INSERT INTO Customers VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        i,
        fake.first_name(),
        fake.last_name(),
        random.choice(["Male","Female"]),
        random.choice(membership_levels),
        fake.date_of_birth(minimum_age=18, maximum_age=70),
        fake.email(),
        income
    ))

conn.commit()

/tmp/ipykernel_783/957950202.py:12: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [12]:
categories = ["Electronics","Clothing","Home","Sports","Books"]

for i in range(1,201):
    cursor.execute("""
    INSERT INTO Products VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        i,
        fake.word().capitalize(),
        random.choice(categories),
        round(random.uniform(5,500),2),
        random.randint(0,1000),
        round(random.uniform(1,5),1),
        round(random.uniform(100,5000),2)
    ))

conn.commit()

In [13]:
order_status = ["Pending","Shipped","Delivered","Cancelled"]

for i in range(1,1501):
    cursor.execute("""
    INSERT INTO Orders VALUES (?, ?, ?, ?, ?, ?)
    """, (
        i,
        random.randint(1,500),
        fake.date_this_year(),
        random.choice(["Standard","Express"]),
        random.choice(order_status),
        round(random.uniform(10,1000),2)
    ))

conn.commit()


/tmp/ipykernel_783/1999344501.py:4: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [14]:
cursor.execute("SELECT COUNT(*) FROM Customers")
print("Customers:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM Products")
print("Products:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM Orders")
print("Orders:", cursor.fetchone()[0])

Customers: 500
Products: 200
Orders: 1500


In [15]:
for order_id in range(1,1501):

    # Each order will have 1 to 5 products
    number_of_items = random.randint(1,5)

    # Random unique products per order
    products_in_order = random.sample(range(1,201), number_of_items)

    for product_id in products_in_order:

        quantity = random.randint(1,5)
        unit_price = round(random.uniform(5,500),2)

        cursor.execute("""
        INSERT INTO Order_Items VALUES (?, ?, ?, ?)
        """, (
            order_id,
            product_id,
            quantity,
            unit_price
        ))

conn.commit()

In [16]:
cursor.execute("SELECT COUNT(*) FROM Order_Items")
print("Order_Items:", cursor.fetchone()[0])

Order_Items: 4479


In [17]:
payment_methods = ["Credit Card","Debit Card","PayPal","Bank Transfer"]
payment_status = ["Completed","Failed","Refunded"]

for i in range(1,1501):

    cursor.execute("""
    INSERT INTO Payments VALUES (?, ?, ?, ?, ?, ?)
    """, (
        i,
        i,  # each payment linked to same order_id
        random.choice(payment_methods),
        str(fake.date_this_year()),
        round(random.uniform(10,1000),2),
        random.choice(payment_status)
    ))

conn.commit()

In [18]:
cursor.execute("SELECT COUNT(*) FROM Payments")
print("Payments:", cursor.fetchone()[0])

Payments: 1500


In [19]:
review_id = 1

for _ in range(800):   # 800 reviews

    cursor.execute("""
    INSERT INTO Reviews VALUES (?, ?, ?, ?, ?, ?)
    """, (
        review_id,
        random.randint(1,200),      # product_id
        random.randint(1,500),      # customer_id
        random.randint(1,5),        # rating (ordinal)
        fake.sentence(),
        str(fake.date_this_year())
    ))

    review_id += 1

conn.commit()

In [20]:
cursor.execute("SELECT COUNT(*) FROM Reviews")
print("Reviews:", cursor.fetchone()[0])

Reviews: 800


In [21]:
# Get 10 existing customer emails
cursor.execute("SELECT email FROM Customers LIMIT 10")
duplicate_emails = cursor.fetchall()

# Assign those emails to random other customers
for i in range(20):
    cursor.execute("""
    UPDATE Customers
    SET email = ?
    WHERE customer_id = ?
    """, (
        duplicate_emails[random.randint(0,9)][0],
        random.randint(1,500)
    ))

conn.commit()

In [22]:
for _ in range(50):  # make 50 reviews missing text
    cursor.execute("""
    UPDATE Reviews
    SET review_text = NULL
    WHERE review_id = ?
    """, (random.randint(1,800),))

conn.commit()

In [23]:
conn.close()